# Evaluación — Módulo 4 · Tema 2: GLM con Python

**Alumno:** Víctor Fernando  
**Variable asignada:** `antiguedad_vehiculo_cat`
**Fecha de entrega:** 09/09/2026

> Se utilizó el dataset `metadatos.json` para el análisis de la variable antiguedad_vehiculo_cat.

---
## Parte 1 · Sesión 1 — Modelo de Frecuencia

**Diagnóstico del supuesto de Poisson (modelo completo):**

| Métrica | Valor |
|---|---:|
| φ de Pearson | 1.1664 |
| Cameron–Trivedi α | 0.0744 |
| z | 15.80 |
| p-value | 3.7 × 10⁻⁵⁶ |

**Rating factors de frecuencia para `antiguedad_vehiculo_cat`:**

| Nivel | RF_frec | IC_inf | IC_sup | p | tasa_emp |
|---|---:|---:|---:|---:|---:|
| (-1, 1] (ref) | 1.0000 | 1.0000 | 1.0000 | — | 0.1698 |
| (1, 2] | 0.7600 | 0.7037 | 0.8207 | < 0.0001 | 0.1290 |
| (10, 15] | 0.8825 | 0.8260 | 0.9428 | 0.0002 | 0.1498 |
| (15, 50] | 0.6795 | 0.6113 | 0.7553 | < 0.0001 | 0.1154 |
| (2, 3] | 0.7254 | 0.6703 | 0.7850 | < 0.0001 | 0.1231 |
| (3, 4] | 0.7923 | 0.7336 | 0.8557 | < 0.0001 | 0.1345 |
| (4, 5] | 0.7934 | 0.7344 | 0.8573 | < 0.0001 | 0.1347 |
| (5, 10] | 0.8216 | 0.7718 | 0.8746 | < 0.0001 | 0.1395 |


### P1. ¿Se cumple la equidispersión? Justifica con φ y con Cameron–Trivedi, y di qué familia usarías.

**Respuesta.** No se cumple estrictamente la equidispersión: φ = 1.1664 > 1 ya que indica que la varianza supera a la media, para cumplirse deberían de ser iguales, al ser mayor es sobredispersión.  
Lo anterior, Cameron–Trivedi lo confirma (α = 0.0744, z = 15.80 y p = 3.7 × 10⁻⁵⁶), por lo que se rechaza la hipótesis de equidispersión, al tener alfa mayor a 0, hay sobredispersión.  
La sobredispersión es **leve** (φ < 1.5); siguiendo el criterio, proponemos utilizar **Regresión Quasi-Poisson** para conservar la media estimada y corregir los errores estándar por √φ, esto corrigería la sobredispersión y se mantendría la interpretabilidad.  
No se requiere Binomial Negativa solo por este diagnóstico; su elección predictiva se evalúa aparte mediante AIC y BIC.

### P2. Interpreta los rating factors: niveles más alto y más bajo, porcentajes, intervalos y significancia.

**Respuesta.** La referencia `(-1, 1]` tiene el RF máximo, 1.0000 (**0% de recargo o descuento**); entre los demás niveles, `(10, 15]` es el mayor con 0.8825, es decir, **11.75% menos frecuencia**.  
El RF mínimo corresponde a `(15, 50]`, con 0.6795: representa un **descuento de 32.05%** en frecuencia respecto de vehículos de hasta un año.  
Ningún IC95% de los niveles no referencia cruza 1 y todos sus p-values son menores que 0.05; los mostrados como cero en la tabla original son valores extremadamente pequeños por redondeo.  
Por ello mantendría los ocho niveles, sujetos al monitoreo habitual de estabilidad y credibilidad.

### P3. ¿Por qué el GLM one-way reproduce la tasa empírica y qué aporta frente a una tabla?

**Respuesta.** En el Poisson one-way con liga log y *offset* de exposición, la ecuación de score de cada nivel impone `Σ(observados − predichos) = 0`.  
Así, la suma predicha coincide con la observada dentro de cada categoría y `exp(β₀ + β_nivel) = Σ siniestros / Σ exposición`, la tasa empírica ponderada.  
La diferencia de 5.2 × 10⁻⁵ es solo precisión numérica.  
GLM combina varias variables de forma multiplicativa, controla sus efectos simultáneamente y aporta IC, pruebas y predicciones; una tabla marginal no lo hace.

---
## Parte 2 · Sesión 2 — Severidad y Selección de Modelos

**Comparación de modelos de frecuencia:**

| Modelo | AIC | BIC | pseudo R² McFadden |
|---|---:|---:|---:|
| Poisson | 125,081.7 | 125,261.7 | 0.0198 |
| Binomial Negativa | 124,925.7 | 125,105.8 | 0.0210 |

**Rating factors de severidad (Gamma) para `antiguedad_vehiculo_cat`:**

| Nivel | RF_sev | severidad_emp |
|---|---:|---:|
| (-1, 1] (ref) | 1.0000 | 1,766 |
| (1, 2] | 0.7141 | 1,261 |
| (10, 15] | 0.7400 | 1,307 |
| (15, 50] | 0.9099 | 1,607 |
| (2, 3] | 0.6865 | 1,212 |
| (3, 4] | 0.6275 | 1,108 |
| (4, 5] | 0.7341 | 1,296 |
| (5, 10] | 0.7342 | 1,297 |

### P4. ¿Por qué se usa Gamma para severidad y no una regresión lineal sobre log(Y)?

**Respuesta.** La Gamma es adecuada para montos positivos, asimétricos y heterocedásticos porque `Var(Y|X) = φμ²`, de modo que el **CV = √φ es constante**, adicionalmente es frecuentemente asimétrica con cola derecha pesada, que puede llegar a suceder con siniestros extraordinarios.  
La Lognormal de `Y` no es un GLM estándar de la familia exponencial natural: su estadístico suficiente involucra `log(y)`, no `y`, asume que su residuo ε sigue una normal, lo cual es inapropiado si los datos están altamente asimñetricos.  
Una regresión lineal de `log(Y)` estima `E[log(Y)|X]`, no `E[Y|X]`, y al volver a pesos necesita una corrección de retransformación por sesgo.  
El Gamma con liga log modela directamente la severidad media `E[Y|X]` en su escala monetaria.
Otras ventajas es que con Gamma tendremos varianza dependiente de la media, predicciones no negativas, modelos o pruebas mñas precisas para tests de bondad de ajuste y la posibilidad de hacer análisis más avanzado como VaR o TVaR para marcos regulatorios.

### P5. ¿Qué modelo elegirías según AIC/BIC y por qué un pseudo R² bajo no implica un mal modelo?

**Respuesta.** **Binomial Negativa**: reduce el AIC en 156.0 puntos (124,925.7 vs. 125,081.7) y el BIC en 155.9 (125,105.8 vs. 125,261.7).  
Aunque el pseudo R² solo pasa de 0.0198 a 0.0210, la ocurrencia de siniestros conserva una gran variación aleatoria irreducible, algo normal en seguros.  
El pseudo R² de McFadden no se interpreta como el R² de una regresión lineal; importan la comparación relativa, la calibración y la discriminación fuera de muestra.
Es mejor elegir un modelo sobre un AIC o BIC bajo ya que con esto indicamos un modelo con un mejor capacidad predictiva, así que al momento de querer estimar deberíamos de observar una mejor precisión. El AIC bajo tiene menos parámetros, lo cual lo hace un modelo más simple y menos propenso a sobrejustarse.

### P6. Compara los rating factors de frecuencia y severidad.

**Respuesta.** Ambos componentes apuntan en la misma dirección: todos los niveles distintos de la referencia tienen RF < 1 tanto en frecuencia como en severidad.  
Sin embargo, difieren en magnitud y orden: `(15, 50]` tiene la frecuencia mínima (0.6795), pero una severidad cercana a la base (0.9099).  
En cambio, `(3, 4]` presenta la severidad mínima (0.6275), mientras su RF de frecuencia es 0.7923.  
Esto justifica modelar **Frecuencia × Severidad por separado**: una variable puede afectar de modo distinto cuántos siniestros ocurren y cuánto cuesta cada uno.

---
## Parte 3 · Sesión 3 — Validación y Tarifa

**Validación out-of-sample del modelo de frecuencia:**

| Métrica | Valor | Ideal |
|---|---:|---:|
| Gini (test) | 0.2315 | > 0.30 aceptable |
| Ratio pred/obs (test) | 1.0249 | ≈ 1.00 |

**Prima pura por nivel de `antiguedad_vehiculo_cat`:**

| Nivel | prima_pura_modelo | factor_tarifa |
|---|---:|---:|
| (-1, 1] (ref) | 305.20 | 1.6738 |
| (1, 2] | 163.76 | 0.8981 |
| (10, 15] | 194.41 | 1.0662 |
| (15, 50] | 186.66 | 1.0237 |
| (2, 3] | 148.41 | 0.8140 |
| (3, 4] | 150.55 | 0.8256 |
| (4, 5] | 176.12 | 0.9659 |
| (5, 10] | 180.37 | 0.9892 |

### P7. Interpreta la calibración y la discriminación fuera de muestra.

**Respuesta.** El ratio pred/obs = 1.0249 indica que el modelo sobrepredice el total observado en **2.49%**, una desviación insignificante, asumimos que está bien calibrado en promedio.  
La calibración mide si el nivel agregado de la predicción coincide con la experiencia observada.  
El Gini = 0.2315 mide la capacidad de **ordenar y separar** riesgos altos de bajos y queda por debajo del referente de 0.30.  
Por tanto, el modelo presenta buena calibración promedio, pero una discriminación **limitada**.

### P8. ¿Qué nivel paga la prima pura más alta y cuál la más baja?

**Respuesta.** La prima pura máxima corresponde a `(-1, 1]`: **305.20** y factor 1.6738, equivalente a un **recargo de 67.38%** sobre la prima promedio, sin considerar el nivel de referencia, sería el rango `(10, 15]` con 163.76 y un recargo del **6.62%**.
La mínima corresponde a `(2, 3]`: **148.41** y factor 0.8140, equivalente a un **descuento de 18.60%**.  
Estos porcentajes se calculan contra la prima media del portafolio.  
La amplitud observada confirma que la antigüedad del vehículo se asocia con diferencias tarifarias materiales en el modelo, así que podemos asumir que la antigüedad tiene un mayor peso que el material del modelo.

### P9. Conclusión de nota técnica

**Respuesta.** La antigüedad del vehículo se asocia con diferencias materiales: todos los niveles distintos de la referencia tienen RF de frecuencia y severidad inferiores a uno.  
Para `(2, 3]`, el RF de frecuencia es 0.7254 (IC95%: 0.6703–0.7850) y el de severidad es 0.6865, con una prima pura de 148.41 unidades monetarias.  
Ese nivel queda 18.60% debajo de la prima promedio, mientras `(-1, 1]` alcanza 305.20 unidades monetarias y un recargo de 67.38%.  
La segmentación Frecuencia × Severidad es actuarialmente sustentable; se recomienda conservarla y monitorear su estabilidad y discriminación fuera de muestra.